In [ ]:
# NOTE : dasch_Lightcurve_Extraction.ipynb - Cell 1
# Batch extraction of DASCH's own light curves.
# Takes the same targets .txt files as the cutout cell (browsed or typed in the boxes at the top of the output) or stars typed in by hand.
# For every star it writes into  data/<Star>/DASCH/  (the star folder is made if it doesn't exist yet):
#   <Star>_dasch_lightcurve_all.csv       every row DASCH returns, with DASCH's standard rejections applied (reject column)
#   <Star>_dasch_lightcurve_accepted.csv  detections that survive the standard rejections (reject == 0 and magcal_magdep present)
#   <Star>_dasch_lightcurve.png           plot of the accepted points, same look as the DASCH panel of the SISaP comparison plot (error bars = magcal_local_error)
#   <Star>_dasch_lightcurve_summary.txt   counts, date range, magnitude range, position check
# Run it on the same kernel as the cutout cell (the one with daschlab).

class _StopCell(Exception):
    def _render_traceback_(self):
        return [str(self)]

_prev_thread = globals().get("_lc_thread")
if _prev_thread is not None and _prev_thread.is_alive():
    raise _StopCell("An earlier run of this cell is still working. Press Pause on that run "
                    "(or wait for it to finish), then re-run this cell.")
_lc_thread = None

from IPython.display import HTML, Image, display
try:
    from daschlab import open_session
except ImportError:
    import sys as _sys
    raise _StopCell(f"daschlab is not installed in this kernel ({_sys.executable}). Switch this notebook to the kernel that "
                    f"01_Load_Cutout uses (kernel picker, top right), or run  %pip install daschlab  in a cell and restart the kernel.")
from pathlib import Path
from datetime import datetime
from importlib import metadata
import html
import re
import shutil
import sys
import threading
import time
import traceback
import numpy as np
import pandas as pd
import ipywidgets as widgets

display(HTML("""
<style>
.output, .output_text, .output_stream, .output_stdout {
    color: white !important;
}
.dark-log-output,
.dark-log-output .jp-OutputArea-output,
.dark-log-output .jp-OutputArea-child,
.dark-log-output .output_area,
.dark-log-output pre {
    background-color: #111 !important;
    color: white !important;
    border: none !important;
}
</style>
"""))

# Paths come from rcb_paths.py in the project root (found by walking up from this notebook's folder; or set the environment variable RCB_PROJECT_ROOT)
import os, sys
from pathlib import Path
_rcb_here = Path.cwd().resolve()
_rcb_cands = ([Path(os.environ['RCB_PROJECT_ROOT'])] if os.environ.get('RCB_PROJECT_ROOT') else []) + [_rcb_here, *_rcb_here.parents]
_rcb_root = next((p for p in _rcb_cands if (p / 'rcb_paths.py').exists()), None)
if _rcb_root is None:
    raise FileNotFoundError(f'rcb_paths.py not found in {_rcb_here} or any folder above it. Put it in the project root, or set RCB_PROJECT_ROOT.')
if str(_rcb_root) not in sys.path:
    sys.path.insert(0, str(_rcb_root))
import importlib, rcb_paths
importlib.reload(rcb_paths)
BASE_DATA_DIR = rcb_paths.DATA_DIR
# Browse for the targets file or type stars in the boxes at the top of the output, then press a Start button.
# TARGETS_FILE is only the file shown in the box when the cell starts.
# Line format (file and manual box): "Name" or "Name|RA|Dec". RA and Dec in decimal degrees, or sexagesimal with RA in hours.
# Explicit coordinates skip the name lookup. Blank lines and lines starting with # are skipped.
TARGETS_FILE = rcb_paths.PHASE_1_FILE

LOG_FILE = BASE_DATA_DIR / f"lightcurve_log_{datetime.now().strftime('%Y%m%d_%H%M%S')}.txt"   # everything the log box shows is also written here

ERR_COL = "magcal_local_error"   # DASCH error column for the error bars; other options: "magcal_local_rms", "magcal_magdep_rms"
SHOW_PLOTS = True                # True: also show each star's plot in the log box when it finishes
STAR_TIMEOUT_SECONDS = 900    # give up on a star if DASCH gives no answer for this long
MAX_REFCAT_SEP_ARCSEC = 5.0   # print a warning if DASCH's lightcurve source is farther than this from the star (None = off); never skips

pause_button = widgets.Button(description="⏸ Pause", button_style="warning", disabled=True,
                               layout=widgets.Layout(width="120px", height="36px"))
progress_html = widgets.HTML(value="<i>Waiting for a Start button...</i>")
top_bar = widgets.HBox([pause_button, progress_html],
                        layout=widgets.Layout(align_items="center", margin="0 0 12px 0"))
log_output = widgets.Output(layout=widgets.Layout(border="1px solid #444", padding="8px",
                                                    max_height="400px", overflow="auto"))
log_output.add_class("dark-log-output")

_lbl = {"description_width": "95px"}
file_text = widgets.Text(value=str(TARGETS_FILE), description="Targets file:", layout=widgets.Layout(width="560px"), style=_lbl)
browse_btn = widgets.Button(description="📂 Browse...", tooltip="Open a file window and pick a targets file from anywhere",
                            layout=widgets.Layout(width="120px", height="36px"))
start_file_btn = widgets.Button(description="▶ Start file", button_style="success", layout=widgets.Layout(width="140px", height="36px"))
manual_text = widgets.Textarea(value="", placeholder="One star per line, e.g.\nUW Cen\nUW Cen|12 43 17.18|-54 31 40.7",
                               description="Manual stars:", layout=widgets.Layout(width="760px", height="84px"), style=_lbl)
start_manual_btn = widgets.Button(description="▶ Start manual", button_style="success", layout=widgets.Layout(width="140px", height="36px"))
overwrite_box = widgets.Checkbox(value=False, description="Download again even if the DASCH folder already has this star's light curve",
                                 indent=False, layout=widgets.Layout(width="620px"))

display(widgets.VBox([widgets.HBox([file_text, browse_btn, start_file_btn]),
                      widgets.HBox([manual_text, start_manual_btn]),
                      overwrite_box,
                      top_bar, log_output]))

pause_event = threading.Event()
pause_button.on_click(lambda b: (pause_event.set(), setattr(pause_button, "description", "Pausing...")))


class StarSkipped(Exception):
    def __init__(self, short, detail):
        super().__init__(detail)
        self.short, self.detail = short, detail


def log(msg=""):
    with log_output:
        print(msg)
    try:
        with open(LOG_FILE, "a", encoding="utf-8") as fh:
            fh.write(str(msg) + "\n")
    except Exception:
        pass

def sanitize_folder_name(name):
    # same rule as the cutout cell, so folders line up with the ones it already made
    cleaned = re.sub(r'[<>:"/\\|?*]', '', name)
    cleaned = cleaned.strip()
    cleaned = re.sub(r'\s+', '_', cleaned)
    return cleaned

def format_eta(seconds):
    if seconds is None or seconds != seconds or seconds < 0:
        return "n/a"
    m, s = divmod(int(seconds), 60)
    h, m = divmod(m, 60)
    return f"{h:d}:{m:02d}:{s:02d}" if h else f"{m:d}:{s:02d}"

def show_message(text, color="#4CAF50"):
    progress_html.value = (f'<div style="font-family: monospace; font-size: 14px; color: {color};">'
                           f'<b>{html.escape(text)}</b></div>')


def parse_object_list_file(path):
    return parse_object_text(Path(path).read_text(encoding="utf-8-sig"))

def parse_object_text(text):
    entries, seen = [], set()
    for line in text.splitlines():
        stripped = line.strip()
        if not stripped or stripped.startswith("#"):
            continue
        parts = [p.strip() for p in stripped.split("|")]
        name, ra_deg, dec_deg = parts[0], None, None
        if len(parts) == 3:
            try:
                ra_deg, dec_deg = float(parts[1]), float(parts[2])
            except ValueError:
                # sexagesimal as SIMBAD prints it: RA in hours "15 48 34.4156", Dec in degrees "+28 09 24.288"
                try:
                    from astropy.coordinates import SkyCoord
                    import astropy.units as u
                    c = SkyCoord(parts[1], parts[2], unit=(u.hourangle, u.deg))
                    ra_deg, dec_deg = float(c.ra.deg), float(c.dec.deg)
                except Exception:
                    ra_deg = dec_deg = None
        elif len(parts) != 1:
            name = stripped
        key = sanitize_folder_name(name).lower()
        if key and key not in seen:
            seen.add(key)
            entries.append((name, ra_deg, dec_deg))
    return entries

def resolve_name_coords(name, tries=3, timeout=30):
    # Name lookup with a longer timeout than daschlab's 10 s, retried a few times
    from astropy.coordinates import SkyCoord
    from astropy.utils.data import conf as _conf
    old, last = _conf.remote_timeout, None
    _conf.remote_timeout = timeout
    try:
        for k in range(tries):
            try:
                print(f"  name lookup attempt {k + 1}/{tries} (waits up to {timeout}s)...")
                c = SkyCoord.from_name(name)
                return float(c.ra.deg), float(c.dec.deg)
            except Exception as e:
                last = e
                print(f"  attempt {k + 1} failed: {type(e).__name__}")
                time.sleep(3 * (k + 1))
    finally:
        _conf.remote_timeout = old
    raise last

def resolve_targets_path(text):
    # The path typed in the box (quotes from "Copy as path" are removed); if it is not there, look for the same file name in the usual folders
    t = text.strip().strip('"').strip("'").strip()
    if not t:
        return None
    p = Path(t)
    if p.is_file():
        return p
    root = BASE_DATA_DIR.parent
    for folder in (TARGETS_FILE.parent, root / "notebooks", root, root / "project_Notes", root / "test_CNN" / "data"):
        if (folder / p.name).is_file():
            return folder / p.name
    return None

def saved_coords(star_dir):
    # position the cutout cell saved for this star (target_coord.txt: "ra dec source"), or None
    try:
        parts = (star_dir / "target_coord.txt").read_text().split()
        return float(parts[0]), float(parts[1])
    except Exception:
        return None

def run_with_timeout(fn, seconds, *args):
    box = {}
    def target():
        try:
            box["value"] = fn(*args)
        except BaseException as e:
            box["error"] = e
    t = threading.Thread(target=target, daemon=True)
    t.start()
    t.join(seconds)
    if t.is_alive():
        raise TimeoutError(f"no answer after {seconds}s")
    if "error" in box:
        raise box["error"]
    return box["value"]


def refcat_offset_arcsec(ra, dec, lc_df):
    # arcsec between the star and the reference-catalog source its lightcurve belongs to (None = couldn't check)
    try:
        import astropy.units as u
        from astropy.coordinates import SkyCoord
        cra = float(lc_df["catalog_ra"].dropna().iloc[0])
        cdec = float(lc_df["catalog_dec"].dropna().iloc[0])
        return float(SkyCoord(ra * u.deg, dec * u.deg).separation(SkyCoord(cra * u.deg, cdec * u.deg)).arcsec)
    except Exception:
        return None

def jd_to_year(jd):
    return 2000.0 + (float(jd) - 2451544.5) / 365.25

def add_jd(lc, lc_df):
    # Make sure lc_df has a time_jd column (daschlab 1.1.0 keeps the time in a 'time' column). Returns how it was found, or None.
    if "time_jd" in lc_df.columns:
        return "time_jd column from DASCH"
    try:
        jd = np.asarray(lc["time"].jd, dtype=float)
        if len(jd) == len(lc_df):
            lc_df["time_jd"] = jd
            return "lc['time'] converted to JD"
    except Exception:
        pass
    if "time" in lc_df.columns:
        s = lc_df["time"]
        try:
            if np.issubdtype(s.dtype, np.datetime64):
                from astropy.time import Time
                lc_df["time_jd"] = Time(s.to_numpy(dtype="datetime64[ns]")).jd
                return "time column (dates) converted to JD"
            v = pd.to_numeric(s, errors="coerce")
            if v.notna().any():
                lc_df["time_jd"] = v + 2400000.5 if v.median() < 1e5 else v
                return "time column (numbers) used as JD"
        except Exception:
            pass
    return None

def make_plot(d, name, path):
    # Same look as the DASCH (bottom) panel of the SISaP comparison plot: black points, red error bars
    from matplotlib.figure import Figure
    d = d[d["time_jd"].notna()]
    err_col = ERR_COL if ERR_COL in d.columns else ("magcal_local_rms" if "magcal_local_rms" in d.columns else None)
    fig = Figure(figsize=(13, 5.5))
    ax = fig.subplots()
    if err_col is not None:
        ax.errorbar(d["time_jd"], d["magcal_magdep"], yerr=d[err_col], fmt="none", ecolor="red", elinewidth=0.7, alpha=0.5, zorder=1)
    ax.scatter(d["time_jd"], d["magcal_magdep"], s=14, c="black", alpha=0.8, zorder=2)
    ax.set_title(f"DASCH: {name} ({len(d)} points" + (f", error bars = {err_col})" if err_col else ")"), fontsize=9)
    ax.set_ylabel("Calibrated magnitude", fontsize=8)
    ax.set_xlabel("Julian Date", fontsize=8)
    ax.grid(alpha=0.3)
    ax.invert_yaxis()
    fig.tight_layout()
    fig.savefig(path, dpi=130)

def build_summary(rec):
    W = 78
    L = ["=" * W, "DASCH LIGHT CURVE EXTRACTION SUMMARY", f"Object : {rec['name']}", "=" * W, ""]
    L.append(f"Extracted at            : {datetime.now().isoformat(timespec='seconds')}")
    L.append(f"Target position (deg)   : RA {rec['ra']:.6f}  Dec {rec['dec']:.6f}   (source: {rec['coord_source']})")
    L.append(f"Reference catalog       : APASS (lightcurve(0) = nearest catalog source)")
    if rec["sep"] is not None:
        L.append(f"Lightcurve source offset: {rec['sep']:.2f} arcsec from the star")
    L.append("")
    L.append(f"Lightcurve rows         : {rec['n_rows']:,}")
    L.append(f"Detected (magcal_magdep): {rec['n_detected']:,}")
    L.append(f"Flagged by standard rejections : {rec['n_rejected']:,}")
    L.append(f"ACCEPTED (detected, not rejected): {rec['n_accepted']:,}")
    if rec["n_accepted"]:
        if "jd_min" in rec:
            L.append(f"Accepted time span      : JD {rec['jd_min']:.1f} - {rec['jd_max']:.1f}   (years {jd_to_year(rec['jd_min']):.1f} - {jd_to_year(rec['jd_max']):.1f})")
        L.append(f"Accepted magnitudes     : brightest {rec['mag_min']:.2f} | median {rec['mag_med']:.2f} | faintest {rec['mag_max']:.2f}")
    L.append("")
    L.append("Files in this folder:")
    for f in rec["files"]:
        L.append(f"  {f}")
    L.append("")
    L.append("DASCH's own warning about apply_standard_rejections(): this is under development, treat with appropriate caution.")
    return "\n".join(L)


def extract_star(rec):
    name, slug = rec["name"], rec["slug"]
    star_dir = BASE_DATA_DIR / slug
    out_dir = star_dir / "DASCH"
    rec["existed_before"] = star_dir.exists()
    acc_path = out_dir / f"{slug}_dasch_lightcurve_accepted.csv"
    all_path = out_dir / f"{slug}_dasch_lightcurve_all.csv"

    if acc_path.exists() and all_path.exists() and not rec["overwrite"]:
        rec["status"] = "already extracted (skipped)"
        log(f"  Already in {out_dir} -- skipped (tick the box at the top to download again).")
        return
    out_dir.mkdir(parents=True, exist_ok=True)

    # coordinates: the line in the file / manual box, else the position the cutout cell saved, else a name lookup
    ra, dec, src = rec["ra"], rec["dec"], "targets list"
    if ra is None or dec is None:
        sc = saved_coords(star_dir)
        if sc is not None:
            ra, dec = sc
            src = "target_coord.txt from the cutout cell"

    # daschlab prints its own progress lines -- run its calls inside log_output so they land in the log box.
    # Output swallows exceptions raised inside it, so catch here and re-raise.
    err = None
    with log_output:
        try:
            sess = open_session(str(out_dir))
            if ra is not None and dec is not None:
                print(f"Using coordinates ra={ra:.5f}, dec={dec:.5f} ({src}) -- skipping name lookup.")
                sess.select_target(ra_deg=ra, dec_deg=dec)
            else:
                try:
                    sess.select_target(name)
                    src = "daschlab name lookup"
                except Exception as e:
                    print(f"  daschlab name lookup failed ({type(e).__name__}); retrying with a longer timeout...")
                    ra, dec = resolve_name_coords(name)
                    src = "name lookup (retry)"
                    print(f"  Resolved {name}: ra={ra:.5f}, dec={dec:.5f}")
                    sess.select_target(ra_deg=ra, dec_deg=dec)
            sess.select_refcat("apass")
            lc = sess.lightcurve(0)
            lc.apply_standard_rejections()
            lc_df = lc.to_pandas()
        except BaseException as e:
            err = e
    if err is not None:
        raise err

    if len(lc_df) == 0 or "magcal_magdep" not in lc_df.columns:
        raise StarSkipped("empty light curve", "DASCH returned no light-curve rows for this position")

    jd_src = add_jd(lc, lc_df)
    if jd_src is None:
        log("  WARNING: couldn't find the observation time in the light curve, so there is no time_jd column (no plot, no date range). "
            f"Columns DASCH returned: {', '.join(map(str, lc_df.columns))}")
    else:
        log(f"  Observation time: {jd_src}")

    if ra is None:
        # name-lookup path inside daschlab: take the catalog position as the nearest thing we have
        try:
            ra, dec = float(lc_df["catalog_ra"].dropna().iloc[0]), float(lc_df["catalog_dec"].dropna().iloc[0])
        except Exception:
            ra = dec = float("nan")
    rec["ra"], rec["dec"], rec["coord_source"] = ra, dec, src

    accepted_df = lc_df[(lc_df["reject"] == 0) & (lc_df["magcal_magdep"].notna())].copy()
    rec["n_rows"] = len(lc_df)
    rec["n_detected"] = int(lc_df["magcal_magdep"].notna().sum())
    rec["n_rejected"] = int((lc_df["reject"] != 0).sum())
    rec["n_accepted"] = len(accepted_df)
    if len(accepted_df):
        m = pd.to_numeric(accepted_df["magcal_magdep"], errors="coerce")
        rec["mag_min"], rec["mag_med"], rec["mag_max"] = float(m.min()), float(m.median()), float(m.max())
        if "time_jd" in accepted_df.columns and accepted_df["time_jd"].notna().any():
            rec["jd_min"], rec["jd_max"] = float(accepted_df["time_jd"].min()), float(accepted_df["time_jd"].max())

    rec["sep"] = None
    if MAX_REFCAT_SEP_ARCSEC is not None and ra == ra:
        rec["sep"] = refcat_offset_arcsec(ra, dec, lc_df)
    if rec["sep"] is None:
        log("  Couldn't check that the lightcurve source sits on the star (continuing).")
    else:
        log(f"  Lightcurve source is {rec['sep']:.1f} arcsec from the star.")
        if rec["sep"] > MAX_REFCAT_SEP_ARCSEC:
            # Warning only: very bright stars have no clean catalog entry, so the nearest source can sit a few arcsec off
            log(f"  WARNING: nearest reference-catalog source is {rec['sep']:.1f} arcsec from the star (warn above {MAX_REFCAT_SEP_ARCSEC}).")

    lc_df.to_csv(all_path, index=False)
    accepted_df.to_csv(acc_path, index=False)
    rec["files"] = [all_path.name, acc_path.name]
    png_path = out_dir / f"{slug}_dasch_lightcurve.png"
    try:
        if "time_jd" not in lc_df.columns:
            raise KeyError("no time_jd column")
        make_plot(accepted_df, name, png_path)
        rec["files"].append(png_path.name)
        if SHOW_PLOTS:
            with log_output:
                display(Image(filename=str(png_path), width=950))
    except Exception as e:
        log(f"  Couldn't make the quick-look plot ({type(e).__name__}: {e}).")
    sum_path = out_dir / f"{slug}_dasch_lightcurve_summary.txt"
    rec["files"].append(sum_path.name)
    sum_path.write_text(build_summary(rec), encoding="utf-8")

    log(f"  Rows {rec['n_rows']:,} | detected {rec['n_detected']:,} | flagged by standard rejects {rec['n_rejected']:,} "
        f"| accepted {rec['n_accepted']:,}")
    log(f"  Saved to {out_dir}")
    rec["status"] = "complete" if rec["n_accepted"] else "complete (no accepted points)"


def new_record(entry, overwrite):
    name, ra, dec = entry
    return {"name": name, "slug": sanitize_folder_name(name), "ra": ra, "dec": dec, "coord_source": "", "overwrite": overwrite,
            "status": "not run", "note": "", "secs": None, "n_rows": 0, "n_detected": 0, "n_rejected": 0, "n_accepted": 0,
            "sep": None, "files": []}

def quick_table(recs):
    L = [f"{'Object':<20}{'Status':<32}{'Rows':>8}{'Detected':>10}{'Accepted':>10}{'Time':>8}"]
    for r in recs:
        L.append(f"{r['name'][:19]:<20}{r['status'][:31]:<32}{r['n_rows']:>8,}{r['n_detected']:>10,}{r['n_accepted']:>10,}"
                 f"{format_eta(r['secs']):>8}")
    return L

def run_batch(entries, source_label, overwrite):
    finish_word, finish_color = "Done", "#4CAF50"
    recs, batch_start = [], time.monotonic()
    try:
        recs.extend(new_record(e, overwrite) for e in entries)
        log(f"{len(recs)} stars from {source_label}: {', '.join(r['name'] for r in recs)}")
        log("(DASCH's own warning about apply_standard_rejections(): this is under development, "
            "treat with appropriate caution)")
        for n, rec in enumerate(recs, 1):
            if pause_event.is_set():
                finish_word, finish_color = "Paused", "#FFA726"
                for rest in recs[n - 1:]:
                    rest["status"] = "not started (paused)"
                break
            log(f"\n[{n}/{len(recs)}] {rec['name']}")
            show_message(f"Star {n}/{len(recs)}: {rec['name']} ...", "#42A5F5")
            t0 = time.monotonic()
            try:
                run_with_timeout(extract_star, STAR_TIMEOUT_SECONDS, rec)
            except StarSkipped as e:
                rec["status"], rec["note"] = f"skipped: {e.short}", e.detail
                log(f"  Skipped -- {e.detail}")
            except TimeoutError:
                rec["status"] = "failed: timed out"
                log(f"  Failed -- no answer from DASCH after {STAR_TIMEOUT_SECONDS}s. Press Start again to retry this star.")
            except Exception as e:
                globals()["last_error_traceback"] = traceback.format_exc()
                rec["status"] = f"failed: {type(e).__name__}"
                rec["note"] = str(e)
                log(f"  Failed -- {type(e).__name__}: {e}")
            rec["secs"] = time.monotonic() - t0
            if rec["status"].startswith(("failed", "skipped")):
                # don't leave an empty folder behind for a star that never got going
                sdir = BASE_DATA_DIR / rec["slug"]
                if not rec.get("existed_before", True) and sdir.exists():
                    try:
                        if not any(p.is_file() for p in sdir.rglob("*")):
                            shutil.rmtree(sdir, ignore_errors=True)
                    except Exception:
                        pass
            log(f"  [{rec['status']}]  {format_eta(rec['secs'])}")
    except Exception:
        globals()["last_error_traceback"] = traceback.format_exc()
        log("Batch stopped by an unexpected error:\n" + traceback.format_exc())
        finish_word, finish_color = "Error", "#E53935"
    finally:
        if recs:
            log("\nSummary:")
            log("\n".join(quick_table(recs)))
            log(f"\nTotal wall-clock: {format_eta(time.monotonic() - batch_start)}")
        pause_button.description = finish_word
        pause_button.disabled = True
        show_message(f"{finish_word} -- {sum(1 for r in recs if r['secs'] is not None)} stars processed", finish_color)
        set_running(False)


def set_running(running):
    start_file_btn.disabled = running
    start_manual_btn.disabled = running
    browse_btn.disabled = running

def begin(entries, source_label):
    global LOG_FILE, _lc_thread
    if _lc_thread is not None and _lc_thread.is_alive():
        log("A run is still going. Wait for it to finish (or press Pause) before starting another.")
        return
    pause_event.clear()
    pause_button.description, pause_button.disabled = "⏸ Pause", False
    LOG_FILE = BASE_DATA_DIR / f"lightcurve_log_{datetime.now().strftime('%Y%m%d_%H%M%S')}.txt"
    set_running(True)
    log(f"\n===== New run: {len(entries)} star(s) from {source_label} =====")
    _lc_thread = threading.Thread(target=run_batch, args=(entries, source_label, bool(overwrite_box.value)), daemon=True)
    _lc_thread.start()

def on_start_file(_=None):
    p = resolve_targets_path(file_text.value)
    if p is None:
        log(f"Targets file not found: {file_text.value.strip()}")
        show_message("Targets file not found", "#E53935")
        return
    try:
        entries = parse_object_list_file(p)
    except Exception as e:
        log(f"Couldn't read {p}: {e}")
        show_message("Couldn't read the targets file", "#E53935")
        return
    if not entries:
        log(f"No star lines found in {p.name}.")
        show_message("No stars in that file", "#E53935")
        return
    begin(entries, str(p))

def on_start_manual(_=None):
    entries = parse_object_text(manual_text.value)
    if not entries:
        log("The manual box is empty. Type one star per line, e.g.  UW Cen   or   UW Cen|12 43 17.18|-54 31 40.7")
        show_message("Nothing typed in the manual box", "#E53935")
        return
    begin(entries, "manual entry")

def on_browse(_=None):
    # Native file window (the kernel runs on this computer, so it can open one); the chosen path goes into the box
    try:
        import tkinter as tk
        from tkinter import filedialog
    except Exception:
        log("The file window isn't available in this Python. Paste the path into the Targets file box instead.")
        return
    cur = Path(file_text.value.strip().strip('"').strip("'") or ".")
    start_dir = cur.parent if cur.parent.is_dir() else Path.home()
    try:
        root = tk.Tk()
        root.withdraw()
        root.attributes("-topmost", True)
        chosen = filedialog.askopenfilename(parent=root, title="Pick a targets file", initialdir=str(start_dir),
                                            filetypes=[("Text files", "*.txt"), ("All files", "*.*")])
        root.destroy()
    except Exception as e:
        log(f"Couldn't open the file window ({type(e).__name__}: {e}). Paste the path into the Targets file box instead.")
        return
    if chosen:
        file_text.value = str(Path(chosen))

start_file_btn.on_click(on_start_file)
start_manual_btn.on_click(on_start_manual)
browse_btn.on_click(on_browse)

# Startup check, printed here in the cell output
BASE_DATA_DIR.mkdir(parents=True, exist_ok=True)
def _ver(pkg):
    try:
        return metadata.version(pkg)
    except Exception:
        return "not found"
print(f"Python    : {sys.executable}")
print(f"daschlab  : {_ver('daschlab')}")
print(f"Data dir  : {BASE_DATA_DIR}   (light curves go to <Star>/DASCH/)")
print(f"Targets   : Browse for a file (or paste its path) and press Start file, or type stars in the Manual box and press Start manual. "
      f"Default file: {TARGETS_FILE} ({'found' if TARGETS_FILE.exists() else 'NOT FOUND'})")